# Demo for Section 3 - Statistics and Metrics

This section demonstrates my `numcompute.stats` and `numcompute.metrics` modules.
The examples cover descriptive statistics, axis behaviour, missing values, histograms, quantiles,
streaming statistics, classification, regression and the bonus ROC/AUC functions.
The last section compares Python loops with the NumPy implementations.



In [39]:
#run this is you do not have numpy installed
%pip install numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [40]:
from pathlib import Path
import sys
import platform
import time
import numpy as np

# Works from either the repository root or its demo folder.
ROOT = Path.cwd()
if not (ROOT / "numcompute").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "numcompute").is_dir():
    raise RuntimeError("Open this notebook from the project root or its demo folder.")
sys.path.insert(0, str(ROOT))
from numcompute import stats, metrics

np.set_printoptions(precision=4, suppress=True)
print("Python:", platform.python_version())
print("NumPy:", np.__version__)


Python: 3.14.3
NumPy: 2.5.4


## 1. Descriptive statistics and missing values
Each row is a student; columns are study hours and score. The second student's hours
are missing. `axis=0` produces one result per column. `nan_policy="omit"` ignores
missing observations: the expected column means are **[4, 70]**.


In [41]:
X = np.array([[2., 50.], [np.nan, 70.], [6., 90.]])
print("Input (3 students, 2 features):\n", X)
for name in ("mean", "median", "std", "min", "max"):
    print(name, getattr(stats, name)(X, axis=0, nan_policy="omit"))
print("Population variance:", stats.var(X, axis=0, nan_policy="omit"))
np.testing.assert_allclose(stats.mean(X, axis=0, nan_policy="omit"), [4, 70])
np.testing.assert_allclose(stats.var(X, axis=0, nan_policy="omit"), [4, 800/3])


Input (3 students, 2 features):
 [[ 2. 50.]
 [nan 70.]
 [ 6. 90.]]
mean [ 4. 70.]
median [ 4. 70.]
std [ 2.     16.3299]
min [ 2. 50.]
max [ 6. 90.]
Population variance: [  4.     266.6667]


## 2. Axis and shape behaviour
`axis=0` reduces rows, `axis=1` reduces columns, and `axis=None` reduces everything.
`keepdims=True` retains the reduced axes with length one. Averaging differently measured
features is shown only to explain shapes, not as a meaningful student measurement.


In [42]:
complete = np.array([[2., 50.], [4., 70.], [6., 90.]])
for axis in (0, 1, None):
    answer = stats.mean(complete, axis=axis)
    print("axis =", axis, "result =", answer, "shape =", np.shape(answer))
kept = stats.mean(complete, axis=0, keepdims=True)
print("keepdims result:", kept, "shape:", kept.shape)
assert kept.shape == (1, 2)
np.testing.assert_allclose(stats.mean(complete, axis=1), [26, 37, 48])


axis = 0 result = [ 4. 70.] shape = (2,)
axis = 1 result = [26. 37. 48.] shape = (3,)
axis = None result = 37.0 shape = ()
keepdims result: [[ 4. 70.]] shape: (1, 2)


## 3. Quantiles and histograms
Quantiles use fractions from 0 to 1. The 0.5 quantile is the median. The missing
value is ignored below. Histogram bins include their left edge; only the last
bin also includes its right edge.


In [43]:
values = [10, 20, np.nan, 30, 40]
for method in ("linear", "lower", "higher", "midpoint"):
    print(method, stats.quantile(values, .5, interpolation=method))
print("Minimum, median, maximum:", stats.quantile(values, [0, .5, 1]))
counts, edges = stats.histogram([50, 60, 70, 80, 90], bins=[0, 60, 80, 100])
print("Histogram counts:", counts)
print("Histogram edges:", edges)
np.testing.assert_array_equal(counts, [1, 2, 2])
assert stats.quantile(values, .5) == 25


linear 25.0
lower 20.0
higher 30.0
midpoint 25.0
Minimum, median, maximum: [10. 25. 40.]
Histogram counts: [1 2 2]
Histogram edges: [  0.  60.  80. 100.]


## 4. Streaming statistics
The Welford accumulator updates its count, mean and variance as batches arrive,
without storing previous batches. Here, **[2, 4]** arrives first, followed by **[6]**.
Population variance divides by count; sample variance uses `ddof=1`.


In [44]:
stream = stats.Welford()
for chunk in ([2, 4], [6]):
    stream.update(chunk)
    print("Batch:", chunk, "count:", stream.count,
          "mean:", stream.mean, "variance:", stream.variance())
print("Sample variance:", stream.variance(ddof=1))
print("Sample standard deviation:", stream.std(ddof=1))
assert stream.count == 3
np.testing.assert_allclose([stream.mean, stream.variance(), stream.std(ddof=1)], [4, 8/3, 2])


Batch: [2, 4] count: 2 mean: 3.0 variance: 1.0
Batch: [6] count: 3 mean: 4.0 variance: 2.6666666666666665
Sample variance: 4.0
Sample standard deviation: 2.0


## 5. Missing values and input validation
By default, a NaN causes the result to be NaN. With `omit`, it is ignored.
With `raise`, missing values are rejected. Empty inputs also raise a clear error.
The current modules use float64 and NumPy reductions; extremely large values can overflow.


In [45]:
print("Propagate:", stats.mean([1, np.nan, 3]))
print("Omit:", stats.mean([1, np.nan, 3], nan_policy="omit"))
assert stats.mean([1, np.nan, 3], nan_policy="omit") == 2
for values, policy in [([], "propagate"), ([1, np.nan], "raise")]:
    try:
        stats.mean(values, nan_policy=policy)
    except ValueError as error:
        print("Expected error:", error)


Propagate: nan
Omit: 2.0
Expected error: data must not be empty
Expected error: data contains NaN


## 6. Classification metrics
The arrays contain actual and predicted outcomes (1 = pass, 0 = fail).
Four of the six predictions are correct. Accuracy, precision, recall and F1
are all 2/3 for this particular example. Confusion-matrix rows are actual labels,
columns are predicted labels, in label order [0, 1].


In [46]:
actual = [1, 1, 1, 0, 0, 0]
predicted = [1, 1, 0, 1, 0, 0]
for name in ("accuracy", "precision", "recall", "f1"):
    result = getattr(metrics, name)(actual, predicted)
    print(name, result)
    np.testing.assert_allclose(result, 2/3)
confusion = metrics.confusion_matrix(actual, predicted)
print("Confusion matrix:\n", confusion)
np.testing.assert_array_equal(confusion, [[2, 1], [1, 2]])


accuracy 0.6666666666666666
precision 0.6666666666666666
recall 0.6666666666666666
f1 0.6666666666666666
Confusion matrix:
 [[2 1]
 [1 2]]


## 7. Multiclass labels and undefined ratios
Use macro, micro or weighted averaging for multiple classes. `average=None`
returns a score per class in sorted label order. A zero denominator returns 0
by default; `zero_division=1` changes that convention.


In [47]:
actual_multi = [0, 0, 1, 1, 2, 2]
predicted_multi = [0, 1, 1, 1, 0, 2]
for averaging in (None, "macro", "micro", "weighted"):
    print("F1", averaging, metrics.f1(actual_multi, predicted_multi, average=averaging))
print("No predicted positives:", metrics.precision([0, 0], [0, 0]))
print("String labels:", metrics.recall(["yes", "yes", "no"],
                                      ["yes", "no", "no"], pos_label="yes"))
np.testing.assert_allclose(metrics.f1(actual_multi, predicted_multi, average=None), [.5, .8, 2/3])


F1 None [0.5    0.8    0.6667]
F1 macro 0.6555555555555556
F1 micro 0.6666666666666666
F1 weighted 0.6555555555555556
No predicted positives: 0.0
String labels: 0.5


## 8. Regression: mean squared error
Actual scores are [50, 70, 90]; predictions are [55, 65, 80]. Squared errors are
[25, 25, 100], whose average is **50**. Identical predictions give zero error.


In [48]:
actual_scores = [50, 70, 90]
predicted_scores = [55, 65, 80]
print("MSE:", metrics.mse(actual_scores, predicted_scores))
print("Perfect predictions:", metrics.mse(actual_scores, actual_scores))
print("Omitting missing pairs:", metrics.mse([1, np.nan, 4], [2, 5, 2], nan_policy="omit"))
assert metrics.mse(actual_scores, predicted_scores) == 50
assert metrics.mse(actual_scores, actual_scores) == 0


MSE: 50.0
Perfect predictions: 0.0
Omitting missing pairs: 2.5


## 9. Bonus: ROC and AUC
ROC evaluates binary prediction scores at different thresholds. Tied scores
are grouped together. A larger score indicates stronger evidence for label 1.
The expected AUC for this example is **0.75**.


In [49]:
fpr, tpr, thresholds = metrics.roc_curve([0, 0, 1, 1], [.1, .4, .35, .8])
print("Threshold    False-positive rate    True-positive rate")
for threshold, false_rate, true_rate in zip(thresholds, fpr, tpr):
    print(f"{threshold:9.2f}    {false_rate:19.2f}    {true_rate:18.2f}")
print("AUC:", metrics.auc(fpr, tpr))
assert metrics.auc(fpr, tpr) == .75


Threshold    False-positive rate    True-positive rate
      inf                   0.00                  0.00
     0.80                   0.00                  0.50
     0.40                   0.50                  0.50
     0.35                   0.50                  1.00
     0.10                   1.00                  1.00
AUC: 0.75


## 10. Python loops vs NumPy
This calls the matching Section 3 benchmark script, so both demos use the same
data generation and timing procedure. Mean, variance, histogram, accuracy, MSE
and confusion matrix outputs are checked against loop implementations before timing.
The table reports the best of five runs after a warm-up. Seed 0 makes input data
reproducible, but timings vary by computer. A speed-up below 1 means NumPy was slower.


In [50]:
import importlib.util
benchmark_path = ROOT / "benchmark" / "benchmark_section3.py"
if not benchmark_path.is_file():
    raise FileNotFoundError("Copy benchmark_section3.py into project benchmark folder first.")
spec = importlib.util.spec_from_file_location("section3_benchmark", benchmark_path)
section3_benchmark = importlib.util.module_from_spec(spec)
spec.loader.exec_module(section3_benchmark)
benchmark_results = section3_benchmark.run_benchmarks(n_rows=20000, repeats=5, seed=0)


Section 3: stats and metrics | rows=20000 | seed=0 | repeats=5
Timing: best of repeats after one warm-up; same repeats for both versions.
Component                Loop (s)    NumPy (s)     Speed-up
------------------------------------------------------------
Mean                     0.000735     0.000014       51.75x
Variance                 0.002849     0.000050       56.54x
Histogram                0.020452     0.000141      145.05x
Accuracy                 0.002652     0.000029       91.45x
MSE                      0.002376     0.000034       68.88x
Confusion matrix         0.003497     0.000263       13.32x

All result comparisons passed.
A speed-up below 1 means NumPy was slower in this run; timings vary by machine.
Environment: Windows-11-10.0.26200-SP0
Processor: Intel64 Family 6 Model 198 Stepping 2, GenuineIntel
Python 3.14.3 | NumPy 2.5.4
